In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn. impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [4]:
df = pd.read_csv('covid_toy.csv')

In [5]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [7]:
df['fever'].value_counts()

fever
101.0    17
98.0     17
104.0    14
100.0    13
99.0     10
102.0    10
103.0     9
Name: count, dtype: int64

In [8]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [6]:
from sklearn.model_selection import train_test_split
X_train , X_test, Y_train , Y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [9]:
X_train

,age,gender,fever,cough,city
44,20,Male,102.0,Strong,Delhi
78,11,Male,100.0,Mild,Bangalore
74,34,Female,104.0,Strong,Delhi
80,14,Female,99.0,Mild,Mumbai
18,64,Female,98.0,Mild,Bangalore
...,...,...,...,...,...
92,82,Female,102.0,Strong,Kolkata
5,84,Female,NaN,Mild,Bangalore
48,66,Male,99.0,Strong,Bangalore
77,8,Female,101.0,Mild,Kolkata


# without column transformation

In [12]:
# adding simple imputer to fever col
si = SimpleImputer()

X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [21]:
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])

X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [16]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first', sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city' ]])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city. shape

(80, 4)

In [18]:
# Extracting Age

X_train_age = X_train.drop(
    columns=['gender', 'fever', 'cough', 'city']
).values

# Also the test data

X_test_age = X_test.drop(
    columns=['gender', 'fever', 'cough', 'city']
).values

X_train_age.shape

(80, 1)

In [23]:
train_transformed = np.concatenate((X_train_age, X_train_fever,X_train_gender_city,X_train_cough), axis=1)
# aLso the test data
test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough), axis=1)

train_transformed. shape

(80, 7)

# with column transformer 

In [24]:
from sklearn.compose import ColumnTransformer

In [32]:
transformer = ColumnTransformer(
    transformers=[
        ('tnf1', SimpleImputer(), ['fever']),
        ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
        ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'city'])
    ],
    remainder='passthrough'
)

In [33]:
transformer.fit_transform(X_train)

array([[102.        ,   1.        ,   1.        ,   1.        ,
          0.        ,   0.        ,  20.        ],
       [100.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  11.        ],
       [104.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  34.        ],
       [ 99.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  14.        ],
       [ 98.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  64.        ],
       [ 98.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  40.        ],
       [ 98.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  69.        ],
       [102.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  24.        ],
       [100.        ,   0.        ,   0.        ,   0.        ,
          1.    

In [34]:
transformer.fit_transform(X_train).shape

(80, 7)

In [35]:
transformer.fit_transform(X_test).shape

(20, 7)